In [1]:
import subprocess
import random
import os
import pandas as pd
import re



In [2]:
outFolder = 'results_20M'
inFolder = 'instances_cpp'

nOfTests = 10

In [3]:
test_name = 'LNS'
file = 'qmcvsbpp'

In [ ]:
for i in range(1, 97):
    in_str = inFolder + f"/instance={i}.txt"

    for test in range(1, nOfTests + 1):

        out_str = outFolder + f"/{test_name}_instance={i}_{test}.txt"

        seed = random.randint(1, 100000)

        proc = subprocess.Popen([f"./{file}", in_str, out_str, str(int(seed))], stdout=subprocess.PIPE)
        output, err = proc.communicate()
        output = output.decode('utf-8')



In [6]:
def df_from_archive(path:str, index:int = 0)-> pd.DataFrame:
    f = open(path, 'r')
    line = f.read()
    line = re.sub(r'\n', r' ', line)

    
    parts = line.split()
    data_dict = {}

    for part in parts:
        if ':' in part:
            key, value = part.split(':')
            data_dict[key] = [txt.strip() for txt in value.split(',') if txt != '']
        
        else:
            data_dict[key].append(part)
    
    
    for key in data_dict.keys():
        if key == 'status':
            data_dict[key] = data_dict[key][0]
        elif key == 'Items' or key == 'Bins':
            data_dict[key] = ''.join([txt for txt in data_dict[key] if txt != ''])
        else:
            data_dict[key] = float(data_dict[key][0])
    
    

    return pd.DataFrame(data_dict, index=[index])
        

In [7]:
final_df = pd.DataFrame()
for i in range(1, 97):
    bestCost = 9999999999
    best_df = pd.DataFrame()

    sum_cost = 0
    sum_time = 0

    for test in range(1, nOfTests + 1):
        ut_str = outFolder + f"/{test_name}_instance={i}_{test}.txt"

        a_df = df_from_archive(ut_str)

        sum_cost += a_df.iloc[0].Cost
        sum_time += a_df.iloc[0].time

        if (a_df.iloc[0].Cost < bestCost):
            bestCost = a_df.iloc[0].Cost
            best_df = a_df
    
    best_df['avgCost'] = float(sum_cost) / float(nOfTests)
    best_df['avgTime'] = float(sum_time) / float(nOfTests) 

    
    
    final_df = pd.concat([final_df, pd.DataFrame(best_df.iloc[0]).T]) 

    
final_df.reset_index(inplace= True, drop= True)
final_df.index = final_df.index + 1


In [8]:
final_df.drop(columns=['Items', 'Bins'], inplace= True)

In [11]:
final_df.Cost.mean()

np.float64(138801.97916666666)